# Lab 4: Memory (Persistence)
Memory allows the graph to persist its state across multiple runs using a checkpointer. This is critical for conversation bots, multi-turn interactions, or workflows that run over time.

We specify memory using `MemorySaver` and execute runs passing a thread configuration: `{"configurable": {"thread_id": "some-id"}}`.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### Define Chat State and compile with Checkpointer

In [2]:
from typing import TypedDict, List, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class ConversationalState(TypedDict):
    messages: Annotated[List[str], operator.add]
    user_input: str

def responder_node(state: ConversationalState):
    print("--- Executing responder_node ---")
    user_msg = state["user_input"]
    bot_reply = f"Processed: {user_msg}"
    # Add both messages to our history list via reducer
    return {"messages": [f"User: {user_msg}", f"Bot: {bot_reply}"]}

conv_builder = StateGraph(ConversationalState)
conv_builder.add_node("responder", responder_node)
conv_builder.add_edge(START, "responder")
conv_builder.add_edge("responder", END)

# Add checkpointer memory
memory = MemorySaver()
conv_graph = conv_builder.compile(checkpointer=memory)

### Test Thread Persistence

# Run under session/thread 1
config1 = {"configurable": {"thread_id": "session-1"}}

print("--- Message 1 (Session 1) ---")
state = conv_graph.invoke({"user_input": "Hello!", "messages": []}, config1)
print("Current History:", state["messages"])

print("\n--- Message 2 (Session 1) ---")
# Note we can pass user_input, and old messages are preserved from the checkpoint!
state = conv_graph.invoke({"user_input": "How do I use memory?", "messages": []}, config1)
print("Current History:", state["messages"])

print("\n--- Message 3 (Session 2 - Brand New Thread) ---")
config2 = {"configurable": {"thread_id": "session-2"}}
state_new = conv_graph.invoke({"user_input": "Hi there!", "messages": []}, config2)
print("Session 2 History:", state_new["messages"])